In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# 1. Load the multimodal feature matrix
input_path = '../data/processed/multimodal_feature_matrix.csv'
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Multimodal matrix not found at {input_path}. Run notebook 07 first.")

df = pd.read_csv(input_path)

# 2. Define Feature Subsets for the Ablation Study
astrometry_cols = ['proper_motion_total', 'tangential_velocity', 'parallax_snr']
photometry_cols = ['absolute_magnitude_g', 'bp_rp']
temporal_cols = ['lc_mean', 'lc_std', 'lc_amplitude', 'lc_skew', 'lc_kurtosis', 'lc_period', 'lc_max_power']

# Ensure any remaining missing temporal data is imputed
df[temporal_cols] = df[temporal_cols].fillna(df[temporal_cols].median())

modalities = {
    'Astrometry': astrometry_cols,
    'Photometry': photometry_cols,
    'Temporal': temporal_cols,
    'Astro_Photo': astrometry_cols + photometry_cols,
    'All_Fused': astrometry_cols + photometry_cols + temporal_cols
}

# 3. Model Initialization
# We use Isolation Forest across all modalities to ensure a controlled ablation baseline
iso_forest = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
scaler = StandardScaler()

def get_anomaly_scores(feature_subset, data):
    X = data[feature_subset].values
    X_scaled = scaler.fit_transform(X)
    
    # Raw decision function: lower/negative values indicate anomalies
    raw_scores = -iso_forest.fit(X_scaled).decision_function(X_scaled)
    
    # Min-Max normalize scores to a unified [0, 1] scale for direct comparison
    return (raw_scores - np.min(raw_scores)) / (np.max(raw_scores) - np.min(raw_scores))

# 4. Execute Multimodal Ablation Study
print("Running anomaly detection across individual and fused modalities...")
for mod_name, cols in modalities.items():
    print(f"Extracting anomaly scores for: {mod_name}...")
    df[f'score_{mod_name}'] = get_anomaly_scores(cols, df)

# 5. Rank Final Candidates based on the fully fused model
df_ranked = df.sort_values(by='score_All_Fused', ascending=False).reset_index(drop=True)

# 6. Save Ablation Results and Final Candidates
output_dir = '../results/candidates'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'multimodal_candidates.csv')
df_ranked.to_csv(output_path, index=False)

print(f"\nMultimodal fusion complete. Candidates saved to {output_path}")
print("\nTop 10 Fused Anomaly Candidates:")
display_cols = ['source_id', 'score_All_Fused', 'score_Astrometry', 'score_Photometry', 'score_Temporal']
display(df_ranked[display_cols].head(10))

Running anomaly detection across individual and fused modalities...
Extracting anomaly scores for: Astrometry...
Extracting anomaly scores for: Photometry...
Extracting anomaly scores for: Temporal...
Extracting anomaly scores for: Astro_Photo...
Extracting anomaly scores for: All_Fused...

Multimodal fusion complete. Candidates saved to ../results/candidates/multimodal_candidates.csv

Top 10 Fused Anomaly Candidates:


,source_id,score_All_Fused,score_Astrometry,score_Photometry,score_Temporal
0,138851516677358336,1.000000,0.852854,0.190282,0.938851
1,138837188665895296,0.978328,0.629276,0.519842,0.827057
2,138844511586077312,0.969342,0.176159,0.104242,0.956399
3,138943978733072768,0.957650,0.152558,0.032821,0.945318
4,138872235599735296,0.954226,0.036035,0.022567,0.975426
5,138940267880957184,0.952603,0.043026,0.866649,0.982006
6,138857843163156736,0.939332,0.739953,0.935516,0.704745
7,138995831873415680,0.927343,0.111851,0.021372,0.923646
8,138862378649972096,0.926817,0.168558,0.757680,0.900936
9,137341681053950592,0.921479,0.326625,0.577578,0.801901
